## Determination of the interaction qubit state

Given a configuration qubit state, one should be able to uniquely determine the interaction qubit state. To do so, we need to loop over different pairs of amino acids in the sequence and compute their distances (see Eq. (29) in SI of Roberts et al. 2021). Considering only the nearest neighbor interactions, if the distance is 1, then we set the corresponding configuration qubit to 1, otherwise to 0.

In [64]:
def generate_full_config_qubit_seq(main_seq: str):
    """
    Generate the full configuration qubit sequence for a given main sequence, which includes filling out the first four fixed qubits corresponding to the first two amino acids, and another fixed qubit at the 6th position. Note that the sequence is in reverse order, i.e. the first two qubit corresponds to the last amino acid in the sequence.

    Args:
        main_seq (str): The qubit sequence corresponding to the main chain, i.e., the output from the qiskit protein folding program, which contains 2 * (N - 3) - 1 qubits for an amino acid sequence of length N.
    """
    # insert a "1" to the left of the last qubit
    new_seq = main_seq[:-1] + "1" + main_seq[-1]
    full_config_seq = new_seq + "0010"
    return full_config_seq


def turn_indicator_funcs(i: int, full_config_seq: str):
    """
    Returns the turn indicator functions for a given amino acid at position i (index starting from 0).
    """
    # Get the reverse string
    full_config_seq = full_config_seq[::-1]
    if i >= len(full_config_seq) / 2:
        raise ValueError(
            f"Amino acid index must be less than {int(len(full_config_seq) / 2)}."
        )
    f0 = (1 - int(full_config_seq[2 * i])) * (1 - int(full_config_seq[2 * i + 1]))
    f1 = int(full_config_seq[2 * i + 1]) * (
        int(full_config_seq[2 * i + 1]) - int(full_config_seq[2 * i])
    )
    f2 = int(full_config_seq[2 * i]) * (
        int(full_config_seq[2 * i]) - int(full_config_seq[2 * i + 1])
    )
    f3 = int(full_config_seq[2 * i]) * int(full_config_seq[2 * i + 1])
    return f0, f1, f2, f3


def compute_distance(i: int, j: int, full_config_seq: str):
    """
    Computes the distance between two amino acids at position i and j (index starting from 0) for a given configuration qubit sequence (each amino acid is represented by two qubits).
    """
    dn0 = sum(
        (-1) ** k * turn_indicator_funcs(k, full_config_seq)[0] for k in range(i, j)
    )
    dn1 = sum(
        (-1) ** k * turn_indicator_funcs(k, full_config_seq)[1] for k in range(i, j)
    )
    dn2 = sum(
        (-1) ** k * turn_indicator_funcs(k, full_config_seq)[2] for k in range(i, j)
    )
    dn3 = sum(
        (-1) ** k * turn_indicator_funcs(k, full_config_seq)[3] for k in range(i, j)
    )
    distance = dn0**2 + dn1**2 + dn2**2 + dn3**2
    return distance


def generate_final_qubit_seq(main_seq: str):
    """
    Generate the final qubit sequence for a given main qubit sequence, including the configuration and interaction qubits.

    Args:
        main_seq (str): The qubit sequence corresponding to the main chain, i.e., the output from the qiskit protein folding program, which contains 2 * (N - 3) - 1 qubits for an amino acid sequence of length N.

    Returns:
        str: The final qubit sequence that can matches the problem Hamiltonian, which contains 2 * (N - 3) - 1 configuration qubits and O(N^2) interaction qubits for an amino acid sequence of length N.
    """
    full_config_seq = generate_full_config_qubit_seq(main_seq)
    full_seq = full_config_seq
    len_peptide = int(len(full_config_seq) / 2) + 1
    for i in range(len_peptide - 4):
        for j in range(i + 5, len_peptide):
            if (j - i) % 2 == 1:
                if compute_distance(i, j, full_config_seq) == 1:
                    full_seq = "1" + full_seq
                else:
                    full_seq = "0" + full_seq
    # remove last four qubits
    final_seq = full_seq[:-4]
    # remove the qubit to the left of the last qubit
    final_seq = final_seq[:-2] + final_seq[-1]
    return final_seq

In [65]:
# 7-AA (zika) example
test_seq = '1000101'
full_test_config_seq = generate_full_config_qubit_seq(test_seq)
print(len(full_test_config_seq))
for i in range(int(len(full_test_config_seq)/2)):
    print(turn_indicator_funcs(i, full_test_config_seq))

12
(0, 1, 0, 0)
(1, 0, 0, 0)
(0, 0, 0, 1)
(0, 1, 0, 0)
(1, 0, 0, 0)
(0, 1, 0, 0)


In [67]:
print(generate_final_qubit_seq(test_seq))
print(len(generate_final_qubit_seq(test_seq)))

011000101
9


In [68]:
# 10-AA (angiotensin) example
test_seq = '1110001110011'
print(generate_final_qubit_seq(test_seq))
print(len(generate_final_qubit_seq(test_seq)))

1000100001110001110011
22
